# 01 — Exploratory Data Analysis & PCA
**Project:** SARS-CoV-2 Genomic Signatures — Explainable Analysis  
**Author:** Ibrahim Mustafa (Bembo)  
**Dataset:** 560,000 genomes × 80 k-mer features (GISAID via SOLE Competition 2025)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy import stats

print('Libraries loaded successfully!')

## 2. Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/your_file.csv', sep='\t')

print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(df.head(2))

## 3. Clean Clade Labels

In [ ]:
df['Clade'] = df['Clade'].str.replace('.', '', regex=False).str.strip()

X = df.drop(columns=['ID', 'Clade'])
y = df['Clade']

print('Clade distribution:')
print(y.value_counts())
print(f'\nFeatures shape: {X.shape}')

## 4. Clade Distribution Plot

In [ ]:
colors = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4','#f032e6']

plt.figure(figsize=(10, 5))
y.value_counts().plot(kind='bar', color=colors)
plt.title('SARS-CoV-2 Clade Distribution (n=560,000)')
plt.xlabel('Clade')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('clade_distribution.png', dpi=150)
plt.show()

## 5. PCA — Raw Data

In [ ]:
sample_idx = df.sample(n=10000, random_state=42).index
X_sample = X.loc[sample_idx]
y_sample = y.loc[sample_idx]

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sample)

plt.figure(figsize=(10, 7))
for i, clade in enumerate(sorted(y_sample.unique())):
    mask = y_sample == clade
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                label=clade, alpha=0.4, s=10, color=colors[i])

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA of SARS-CoV-2 Genomic Signatures (Raw)')
plt.legend(markerscale=3)
plt.tight_layout()
plt.savefig('pca_raw.png', dpi=150)
plt.show()

print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%')

## 6. Outlier Removal & Clean PCA

In [ ]:
z_scores = np.abs(stats.zscore(X_sample))
mask_clean = (z_scores < 3).all(axis=1)

X_clean = X_sample[mask_clean]
y_clean = y_sample[mask_clean]

print(f'Before: {X_sample.shape[0]} | After: {X_clean.shape[0]} | Removed: {X_sample.shape[0]-X_clean.shape[0]}')

pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_clean)

plt.figure(figsize=(10, 7))
for i, clade in enumerate(sorted(y_clean.unique())):
    mask = y_clean == clade
    plt.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                label=clade, alpha=0.4, s=10, color=colors[i])

plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA of SARS-CoV-2 Genomic Signatures (Cleaned)')
plt.legend(markerscale=3)
plt.tight_layout()
plt.savefig('pca_clean.png', dpi=150)
plt.show()

print(f'Variance explained: PC1={pca2.explained_variance_ratio_[0]*100:.1f}%, PC2={pca2.explained_variance_ratio_[1]*100:.1f}%')